# Spatial Detector Comparison — per-scenario

Compares **DSM** (our global), **CF-Attn / CF-Attn-CFAR / NeighborMLP / NeighborMLP-CFAR** (spatial DSMs),
**AMF** + **AMF-local**, **GMM-Levin**.

Each scenario has its own cell — edit its `OVERRIDES` dict then run to train + score + plot.

Scenarios: **0–3** = manual boxes, **4–7** = random boxes. Deep nets train on **GPU**.

To re-extract plots from a saved run (no retraining) see the **Re-extract** cell at the bottom.

In [ ]:
# ── Cell 1: Clone repo + install deps ─────────────────────────────────
import os, sys

REPO_URL      = 'https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL_PROJECT = '/content/proj'
BRANCH        = 'iid-unified-experiment'

if os.path.exists(os.path.join(LOCAL_PROJECT, '.git')):
    !git -C {LOCAL_PROJECT} fetch --all -q
    !git -C {LOCAL_PROJECT} checkout {BRANCH} -q
    !git -C {LOCAL_PROJECT} pull origin {BRANCH} -q
else:
    !git clone -b {BRANCH} --depth 1 {REPO_URL} {LOCAL_PROJECT}

!pip install -q scikit-learn scipy tqdm matplotlib pyyaml pandas

for p in [LOCAL_PROJECT, os.path.join(LOCAL_PROJECT, 'experiments', 'spatial')]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(LOCAL_PROJECT)
print('Ready. CWD:', os.getcwd())
!git -C {LOCAL_PROJECT} log --oneline -1

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU — enable Runtime > Change runtime type > GPU.')
print('PyTorch', torch.__version__, '  device =', DEVICE)

In [ ]:
# ── Cell 3: Mount Drive + link pavia-u.mat ────────────────────────────
import os
RESULTS = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/final_paper/compare_results'
    os.makedirs('/content/proj/data', exist_ok=True)
    DST = '/content/proj/data/pavia-u.mat'
    if not os.path.exists(DST):
        for src in ['/content/drive/MyDrive/final_paper/pavia-u.mat',
                    '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat']:
            if os.path.exists(src):
                os.symlink(src, DST); print('Linked dataset from', src); break
        else:
            print('WARNING: pavia-u.mat not found on Drive.')
    else:
        print('Dataset already linked.')
except Exception as e:
    print('Drive not mounted:', repr(e))
os.makedirs(RESULTS, exist_ok=True)
assert os.path.exists('/content/proj/data/pavia-u.mat'), 'pavia-u.mat missing!'
print('RESULTS dir:', RESULTS)

In [ ]:
# ── Cell 4: Helpers ───────────────────────────────────────────────────
import sys, os, glob, json, importlib, tempfile
import yaml
import numpy as np, pandas as pd
from IPython.display import display, Image as IPImage

QUICK = False   # True = fast smoke test (few epochs)

# Base config — loaded from colab.yaml; per-scenario OVERRIDES are merged on top.
BASE_CFG = yaml.safe_load(open('experiments/spatial/colab.yaml')) or {}

# Figures produced per detection run (in display order).
FIGS = ['false_color', 'label_map_targets', 'signatures',
        'detection_maps', 'detected_pfa',
        'false_alarms_falsecolor', 'false_alarms_labelmap',
        'roc', 'pfa_per_class']

def _reload_modules():
    """Reload full project chain leaf-first so a git pull is always picked up
    without restarting the runtime."""
    for _m in ['dsm_model', 'cfattn_model', 'neighbor_mlp_model', 'local_detectors',
               'final_paper_experiments.models.neighbor_adapted',
               'run_colab', 'run_compare']:
        if _m in sys.modules:
            importlib.reload(sys.modules[_m])

def _latest_run_for(scenario_idx):
    cands = []
    for r in glob.glob(os.path.join(RESULTS, 'compare_*')):
        mp = os.path.join(r, 'metrics.json')
        if os.path.exists(mp):
            try:
                if json.load(open(mp)).get('scenario_index') == scenario_idx:
                    cands.append(r)
            except Exception:
                pass
    return max(cands) if cands else None

def _show_one(sub, label):
    csv = os.path.join(sub, 'summary_table.csv')
    if not os.path.exists(csv):
        return
    print('\n' + '=' * 70); print(label, '::', sub); print('=' * 70)
    df = pd.read_csv(csv)
    pfa_cols = [c for c in df.columns if c.startswith('Pfa')]
    display(df.style
            .background_gradient(subset=['pAUC@0.05', 'AUC', 'Pd@Pfa=0.05'], cmap='Greens')
            .background_gradient(subset=pfa_cols, cmap='Reds')
            .format({c: '{:.3f}' for c in df.columns if c != 'Detector'}))
    for name in FIGS:
        png = os.path.join(sub, name + '.png')
        if os.path.exists(png):
            print(name)
            display(IPImage(filename=png, width=1000))

def show_results(run_dir):
    if run_dir is None:
        print('No run dir found.'); return
    print('Run:', run_dir)
    _show_one(run_dir, 'IN-PATCH signature')
    _show_one(os.path.join(run_dir, 'foreign'), 'FOREIGN signature')

def run_scenario(scenario_idx, overrides=None, show=True):
    """Train all models for one scenario and display results.
    Reloads the full module chain so git pull is always picked up."""
    cfg = dict(BASE_CFG)
    cfg.update(overrides or {})
    cfg['scenario_index'] = int(scenario_idx)
    fd, tmp = tempfile.mkstemp(suffix='.yaml', prefix=f'cfg_s{scenario_idx}_')
    os.close(fd)
    yaml.safe_dump(cfg, open(tmp, 'w'))
    argv = ['run_compare.py', '--config', tmp,
            '--results_dir', RESULTS, '--scenario', str(scenario_idx)]
    if QUICK:
        argv.append('--dry-run')
    print('Running scenario', scenario_idx, '| overrides:', overrides or {}, flush=True)
    _reload_modules()
    import run_compare
    _argv = sys.argv; sys.argv = argv
    try:
        run_compare.main()
    except SystemExit:
        pass
    finally:
        sys.argv = _argv
    run_dir = _latest_run_for(scenario_idx)
    if show:
        show_results(run_dir)
    return run_dir

def extract_from_models(models_pt, overrides=None, show=True):
    """Load a saved models.pt and re-extract ALL plots/tables — no retraining.
    Only scoring/plot knobs can be changed (cfar_*, pfa_target, amplitude, ...).
    Architecture and box params always come from the checkpoint."""
    cfg = dict(BASE_CFG); cfg.update(overrides or {})
    fd, tmp = tempfile.mkstemp(suffix='.yaml', prefix='cfg_reload_'); os.close(fd)
    yaml.safe_dump(cfg, open(tmp, 'w'))
    argv = ['run_compare.py', '--from-models', models_pt,
            '--config', tmp, '--results_dir', RESULTS]
    print('Re-extracting from', models_pt, '| overrides:', overrides or {}, flush=True)
    _reload_modules()
    import run_compare
    _argv = sys.argv; sys.argv = argv
    try:
        run_compare.main()
    except SystemExit:
        pass
    finally:
        sys.argv = _argv
    run_dir = sorted(glob.glob(os.path.join(RESULTS, 'compare_*')))[-1]
    if show:
        show_results(run_dir)
    return run_dir

print('Helpers ready.  QUICK =', QUICK)

## Scenario 0 — manual box 0

In [ ]:
# ── Scenario 0 ────────────────────────────────────────────────────────
OVERRIDES = dict(
    # amplitude         = 0.15,
    # target_fraction   = 0.10,
    # k                 = 5,
    # cfar_bg_window    = 11,
    # cfar_guard        = 1,
    # cfar_lam          = 0.0,
    # cfattn_epochs     = 1000,
    # nmlp_epochs       = 1000,
    # dsm_epochs        = 1000,
    # nmlp_d_lat        = 16,
    # nmlp_enc_hidden   = [128, 64],
    # nmlp_score_hidden = [128],
)
run_dir_0 = run_scenario(0, OVERRIDES)

## Scenario 1 — manual box 1

In [ ]:
# ── Scenario 1 ────────────────────────────────────────────────────────
OVERRIDES = dict(
    # amplitude         = 0.15,
    # target_fraction   = 0.10,
    # k                 = 5,
    # cfar_bg_window    = 11,
    # cfar_guard        = 1,
    # cfar_lam          = 0.0,
    # cfattn_epochs     = 1000,
    # nmlp_epochs       = 1000,
    # dsm_epochs        = 1000,
    # nmlp_d_lat        = 16,
    # nmlp_enc_hidden   = [128, 64],
    # nmlp_score_hidden = [128],
)
run_dir_1 = run_scenario(1, OVERRIDES)

## Scenario 2 — manual box 2

In [ ]:
# ── Scenario 2 ────────────────────────────────────────────────────────
OVERRIDES = dict(
    # amplitude         = 0.15,
    # target_fraction   = 0.10,
    # k                 = 5,
    # cfar_bg_window    = 11,
    # cfar_guard        = 1,
    # cfar_lam          = 0.0,
    # cfattn_epochs     = 1000,
    # nmlp_epochs       = 1000,
    # dsm_epochs        = 1000,
    # nmlp_d_lat        = 16,
    # nmlp_enc_hidden   = [128, 64],
    # nmlp_score_hidden = [128],
)
run_dir_2 = run_scenario(2, OVERRIDES)

## Scenario 3 — manual box 3

In [ ]:
# ── Scenario 3 ────────────────────────────────────────────────────────
OVERRIDES = dict(
    # amplitude         = 0.15,
    # target_fraction   = 0.10,
    # k                 = 5,
    # cfar_bg_window    = 11,
    # cfar_guard        = 1,
    # cfar_lam          = 0.0,
    # cfattn_epochs     = 1000,
    # nmlp_epochs       = 1000,
    # dsm_epochs        = 1000,
    # nmlp_d_lat        = 16,
    # nmlp_enc_hidden   = [128, 64],
    # nmlp_score_hidden = [128],
)
run_dir_3 = run_scenario(3, OVERRIDES)

## Scenario 4 — random box (seed 42)

In [ ]:
# ── Scenario 4 ────────────────────────────────────────────────────────
OVERRIDES = dict(
    # amplitude         = 0.15,
    # target_fraction   = 0.10,
    # k                 = 5,
    # cfar_bg_window    = 11,
    # cfar_guard        = 1,
    # cfar_lam          = 0.0,
    # cfattn_epochs     = 1000,
    # nmlp_epochs       = 1000,
    # dsm_epochs        = 1000,
    # nmlp_d_lat        = 16,
    # nmlp_enc_hidden   = [128, 64],
    # nmlp_score_hidden = [128],
)
run_dir_4 = run_scenario(4, OVERRIDES)

## Scenario 5 — random box (seed 123)

In [ ]:
# ── Scenario 5 ────────────────────────────────────────────────────────
OVERRIDES = dict(
    # amplitude         = 0.15,
    # target_fraction   = 0.10,
    # k                 = 5,
    # cfar_bg_window    = 11,
    # cfar_guard        = 1,
    # cfar_lam          = 0.0,
    # cfattn_epochs     = 1000,
    # nmlp_epochs       = 1000,
    # dsm_epochs        = 1000,
    # nmlp_d_lat        = 16,
    # nmlp_enc_hidden   = [128, 64],
    # nmlp_score_hidden = [128],
)
run_dir_5 = run_scenario(5, OVERRIDES)

## Scenario 6 — random box (seed 456)

In [ ]:
# ── Scenario 6 ────────────────────────────────────────────────────────
OVERRIDES = dict(
    # amplitude         = 0.15,
    # target_fraction   = 0.10,
    # k                 = 5,
    # cfar_bg_window    = 11,
    # cfar_guard        = 1,
    # cfar_lam          = 0.0,
    # cfattn_epochs     = 1000,
    # nmlp_epochs       = 1000,
    # dsm_epochs        = 1000,
    # nmlp_d_lat        = 16,
    # nmlp_enc_hidden   = [128, 64],
    # nmlp_score_hidden = [128],
)
run_dir_6 = run_scenario(6, OVERRIDES)

## Scenario 7 — random box (seed 789)

In [ ]:
# ── Scenario 7 ────────────────────────────────────────────────────────
OVERRIDES = dict(
    # amplitude         = 0.15,
    # target_fraction   = 0.10,
    # k                 = 5,
    # cfar_bg_window    = 11,
    # cfar_guard        = 1,
    # cfar_lam          = 0.0,
    # cfattn_epochs     = 1000,
    # nmlp_epochs       = 1000,
    # dsm_epochs        = 1000,
    # nmlp_d_lat        = 16,
    # nmlp_enc_hidden   = [128, 64],
    # nmlp_score_hidden = [128],
)
run_dir_7 = run_scenario(7, OVERRIDES)

## Re-extract plots from a saved run (no retraining)

In [ ]:
# ── Re-extract all plots from a saved run directory ───────────────────
# Point RUN_DIR to any compare_* folder that contains models.pt.
# Useful to change CFAR params, pfa_target, amplitude, etc. without retraining.
# NOTE: architecture params (nmlp_enc_hidden, cfattn_h, ...) are IGNORED here —
#       they always come from the checkpoint.  Only scoring/plot knobs apply.

RUN_DIR = '/content/drive/MyDrive/final_paper/compare_results/compare_XXXXXXXX_XXXXXX'

OVERRIDES = dict(
    # cfar_bg_window  = 11,
    # cfar_guard      = 1,
    # cfar_lam        = 0.0,
    # pfa_target      = 0.05,
    # amplitude       = 0.15,
    # target_fraction = 0.10,
    # ridge_n_mc      = 8,
)

models_pt = os.path.join(RUN_DIR, 'models.pt')
assert os.path.exists(models_pt), f'models.pt not found in {RUN_DIR}'
new_run_dir = extract_from_models(models_pt, overrides=OVERRIDES)

## Aggregate — mean across all ran scenarios

In [ ]:
# ── Aggregate: mean of each metric across whichever scenarios you ran ──
import glob, json
import numpy as np, pandas as pd
from IPython.display import display

by_scen = {}
for r in sorted(glob.glob(os.path.join(RESULTS, 'compare_*'))):
    mp = os.path.join(r, 'metrics.json')
    if os.path.exists(mp):
        try:
            m = json.load(open(mp))
            sidx = m.get('scenario_index')
            if sidx is not None:
                by_scen[sidx] = m   # keep latest run per scenario
        except Exception:
            pass
print('Scenarios aggregated:', sorted(by_scen))

metrics = ['pAUC@0.05', 'AUC', 'Pd@Pfa=0.05', 'Pfa_avg', 'Pfa_max']
if by_scen:
    dets = [row['Detector'] for row in next(iter(by_scen.values()))['rows']]
    acc = {d: {mt: [] for mt in metrics} for d in dets}
    for m in by_scen.values():
        for row in m['rows']:
            for mt in metrics:
                acc[row['Detector']][mt].append(float(row[mt]))
    rows = []
    for d in dets:
        rec = {'Detector': d, 'n_scen': len(acc[d]['AUC'])}
        for mt in metrics:
            rec[mt] = float(np.nanmean(acc[d][mt]))
        rows.append(rec)
    df = pd.DataFrame(rows)
    display(df.style
            .background_gradient(subset=['pAUC@0.05', 'AUC', 'Pd@Pfa=0.05'], cmap='Greens')
            .background_gradient(subset=['Pfa_avg', 'Pfa_max'], cmap='Reds')
            .format({c: '{:.3f}' for c in metrics}))
    out_csv = os.path.join(RESULTS, 'summary_all_scenarios.csv')
    df.to_csv(out_csv, index=False)
    print('Saved:', out_csv)
else:
    print('No scenario results found yet.')